# toyaikit: Auto Tool Generation from Docstrings

Introduces `toyaikit`, a helper library that removes the boilerplate from the agentic loop.
Instead of writing JSON tool definitions by hand, toyaikit reads your function docstrings and
type hints and generates them automatically.

## Concepts covered
- **Auto tool generation**: `Tools.add_tool()` reads docstrings → produces JSON tool definitions
- **`add_tool` vs `add_tools`**: register one function vs all methods from a class instance
- **`OpenAIResponsesRunner`**: replaces the hand-written `while True` loop
- **Class-based tools**: grouping related tools with shared state (e.g. the index) into a class

In [1]:
!uv add toyaikit

Resolved 152 packages in 707ms                                       
Prepared 4 packages in 248ms                                             
Installed 4 packages in 12ms                                
 + anthropic==0.97.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.57
 + toyaikit==0.0.10


In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [11]:
instructions = """
You're a documentation assistant.

Answer the user question using the documentation knowledge base

Make 3 iterations:
1) in the first iteration, perform one search
2) in the second iteration, analyze the results from the previous search
    and perform 2 more searches
3) syntesize the results into the output

IMPORTANT: At each step, give an explanation of why you want to perform
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.


Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, 
so you don't need to include the word 'evidently'
in search results
"""

In [3]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function", 
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [4]:
def add_entry(filename, title, description, content):
    entry = {'start': 0,
              'content': content,
              'title': title,
              'description': description,
              'filename': filename
             }

    index.append(entry)
    return "OK"

add_entry_tool ={
   "type": "function",
   "name": "add_entry",
   "description": "Add a new entry with metadata and content to an index.",
   "parameters": {
      "type": "object",
      "properties": {
         "filename": {
            "type": "string",
            "description": "The name of the file associated with the entry"
         },
         "title": {
            "type": "string",
            "description": "The title of the entry"
         },
         "description": {
            "type": "string",
            "description": "A short description of the entry"
         },
         "content": {
            "type": "string",
            "description": "The main content of the entry"
         }
      },
      "required": [
         "filename",
         "title",
         "description",
         "content"
      ]
   }
}


In [6]:
from toyaikit.tools import Tools

## Part 1: toyaikit with Hand-Written Tool Definitions

First approach: still write JSON tool definitions manually, but use toyaikit's `Tools`
and `OpenAIResponsesRunner` to handle the loop. The hand-written defs are passed alongside
the function: `add_tool(fn, tool_definition)`.

In [8]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

In [13]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(model='gpt-4o-mini', client=OpenAI())

In [14]:
agent = OpenAIResponsesRunner(tools=agent_tools, 
                              developer_prompt=instructions,
                              # chat_interface=,
                              llm_client=llm_client)

In [15]:
result = agent.loop('how can I create evidently dshaborad')

In [17]:
print(result.last_message)

The final search on "Evidently dashboard examples" yielded valuable information regarding how to implement specific metrics within panels. For instance, examples demonstrate how to use the `UniqueValueCount` metric or the `MinValue` metric from a TextEvals preset. This shows practical usage of metrics and how to configure panels to visualize these data points effectively.

### Summary for Creating an Evidently Dashboard:

1. **Adding Tabs and Panels**:
   - Organize your dashboard using tabs. Multiple tabs can be created in edit mode, allowing you to categorize your panels effectively.
   - Use the option to add various types of panels, including counters, pie charts, and line plots.

2. **Panel Configuration**:
   - Each panel can display data based on selected metrics. You can configure parameters such as title, size, and plot type.
   - Ensure that the metrics you're referencing are present in the reports logged within your project. This is vital for the panels to be populated with 

In [18]:
result.cost.total_cost

Decimal('0.00423585')

In [21]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [22]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

In [23]:
result = agent.loop(
    'how can I create evidently dashbaxrd',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


Parameter,Type,Description
title,str,Title of the panel.
description,str,Optional subtitle.
size,str,"Panel size (""full"" or ""half"")."
values,list,List of metrics to display.
tab,str,Name of the tab for this panel.
create_if_not_exists,bool,Whether to create the tab if it doesn't exist.
plot_params,dict,"Visualization settings like plot type (e.g., ""line"")."


In [25]:
result2 = agent.loop(
    'show  me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)

-> Response received


In [26]:
result2.new_messages

[EasyInputMessage(content='show  me the code', role='user', phase=None, type=None),
 ResponseOutputMessage(id='msg_0ecd38d594dbc8b80069f6aaaab30081979bee0e18b71e0ab7', content=[ResponseOutputText(annotations=[], text='Here’s the code to create a dashboard using the Python API, complete with a text panel, a counter, a line plot, and a pie chart:\n\n### Example Code to Create a Dashboard\n\n```python\n# Import necessary modules\nfrom evidently.sdk.models import PanelMetric\nfrom evidently.sdk.panels import DashboardPanelPlot\nfrom evidently.sdk import Project\n\n# Connect to your project (Replace with your connection method)\nproject = Project("Your_Project_Name")  # Make sure to set up your project\n\n# Create a new dashboard tab\nproject.dashboard.add_tab("My New Tab")\n\n# Add a Text Panel\nproject.dashboard.add_panel(\n    DashboardPanelPlot(\n        title="Dashboard Overview",\n        size="full",\n        values=[],  # Empty for text panel\n        plot_params={"plot_type": "text

In [27]:
agent = OpenAIResponsesRunner(tools=agent_tools, 
                              developer_prompt=instructions,
                              chat_interface=chat_interface,
                              llm_client=llm_client)

In [28]:
result = agent.run()

You: dashboard pls


-> Response received


-> Response received


-> Response received


You: show me the code


-> Response received


You: show me more code


-> Response received


You: stop


Chat ended.


In [30]:
result.cost.total_cost

Decimal('0.00786315')